In [1]:
# ============================================================================
# 0) 导入依赖
# ============================================================================
from datetime import datetime
from pathlib import Path
from dateutil.relativedelta import relativedelta

import polars as pl
from tqdm import tqdm

from vnpy.trader.constant import Interval, Exchange, FactorType
from vnpy.trader.object import FactorRequest
from vnpy.alpha.lab import AlphaLab
from vnpy.alpha.logger import logger

# 因子定义注册表
from vnpy.factor_define import (
    FACTOR_REGISTRY,
    PARAMS_REGISTRY,
    FACTOR_NAMES,
)


In [2]:
# ============================================================================
# 1) 全局参数配置
# ============================================================================

# -- 指数与路径 --
VT_INDEX_SYMBOL: str = '000300.SSE'
BASE_PATH: Path = Path('D:/Aquant project/MF')
LAB_PATH: Path = BASE_PATH / 'MF_lab'

# -- 计算时间范围 --
START: datetime = datetime(2018, 1, 1)
END: datetime = datetime(2026, 5, 10)

# -- 待计算因子列表 --
# 全部注册因子
FACTORS: list[str] = FACTOR_NAMES
# FACTORS = ['streverse_2m']  # <-- 调试单因子时取消注释

# -- 因子类型（统一使用 PRICE_VOLUME，基本面因子请改为 FUNDAMENTAL） --
DEFAULT_FACTOR_TYPE: FactorType = FactorType.PRICE_AND_VOLUME

logger.info(f'计算因子数量: {len(FACTORS)}，时间范围: {START.date()} ~ {END.date()}')

2026-05-22 21:06:39 计算因子数量: 65，时间范围: 2018-01-01 ~ 2026-05-10


In [3]:
# ============================================================================
# 2) 初始化 AlphaLab
# ============================================================================

lab = AlphaLab(str(LAB_PATH))
logger.info(f'Lab 路径: {LAB_PATH}')

2026-05-22 21:06:41 Lab 路径: D:\Aquant project\MF\MF_lab


In [6]:
# ============================================================================
# 3) 工具函数：获取指定年月的起止时间
# ============================================================================

def get_month_start_end(year: int, month: int, current_date: datetime | None = None) -> tuple[datetime, datetime]:
    """
    返回指定年月的第一天 00:00:00 和最后一天 23:59:59。
    若 current_date 落在该月，则 end 截断至 current_date。
    """
    start = datetime(year, month, 1)

    if month == 12:
        normal_end = datetime(year + 1, 1, 1) - relativedelta(days=1)
    else:
        normal_end = datetime(year, month + 1, 1) - relativedelta(days=1)

    if current_date and current_date.year == year and current_date.month == month:
        end = min(normal_end, current_date)
        logger.info(f'    月末截断: {normal_end.date()} -> {end.date()}')
    else:
        end = normal_end

    start = start.replace(hour=0, minute=0, second=0)
    end = end.replace(hour=23, minute=59, second=59)

    return start, end


def generate_year_months(start: datetime, end: datetime) -> list[tuple[int, int]]:
    """生成从 start 到 end 的所有 (year, month) 组合"""
    result: list[tuple[int, int]] = []
    current = start.replace(day=1)
    while current <= end:
        result.append((current.year, current.month))
        current += relativedelta(months=1)
    return result


years_months = generate_year_months(START, END)
logger.info(f'共 {len(years_months)} 个月待计算')

2026-05-22 18:10:30 共 101 个月待计算


In [7]:
# ============================================================================
# 4) 单月因子计算函数
# ============================================================================

def cal_factors_for_month(
    year: int,
    month: int,
    lab: AlphaLab,
    factor_names: list[str],
    index_symbol: str,
    current_date: datetime,
    factor_type: FactorType = FactorType.PRICE_AND_VOLUME,
) -> None:
    """
    计算指定月份的所有因子并保存。
    
    流程:
      1. 获取该月时间范围
      2. 加载当月（或前月）成分股
      3. 逐个因子构造 FactorRequest -> cal_factor_daliy -> save_factor
    """
    year_month = f'{year}-{month:02d}'
    month_start, month_end = get_month_start_end(year, month, current_date)
    logger.info(f'[{year_month}] 时间范围: {month_start.date()} ~ {month_end.date()}')

    # 加载当月成分股
    symbols = lab.load_component_symbols(index_symbol, month_start, month_end)
    if not symbols:
        # 若当月无成分股数据，回退到前一个月
        prev = month_end - relativedelta(months=1)
        prev_start, prev_end = get_month_start_end(prev.year, prev.month, current_date)
        symbols = lab.load_component_symbols(index_symbol, prev_start, prev_end)
        logger.warning(f'[{year_month}] 无当月成分股，回退至 {prev.year}-{prev.month:02d}，共 {len(symbols)} 只')
    else:
        logger.info(f'[{year_month}] 成分股数量: {len(symbols)}')

    # 拆分 symbol / exchange 用于 FactorRequest
    raw_symbols: list[str] = []
    exchanges: list[Exchange] = []
    for vt_symbol in symbols:
        sym, exc = vt_symbol.split('.')
        raw_symbols.append(sym)
        exchanges.append(Exchange(exc))

    ok_count = 0
    for i, factor_name in enumerate(factor_names, 1):
        try:
            req = FactorRequest(
                is_FD=True,
                start=month_start,
                end=month_end,
                symbols=raw_symbols,
                exchanges=exchanges,
                factor_name=factor_name,
                factor_type=factor_type,
            )
            factor_data = lab.cal_factor_daliy(req)

            if not factor_data:
                logger.warning(f'  [{i}/{len(factor_names)}] {factor_name}: 无结果')
                continue

            lab.save_factor(factor_data, recalcu=True)
            ok_count += 1
            logger.info(f'  [{i}/{len(factor_names)}] {factor_name}: 已保存')
        except Exception as e:
            logger.error(f'  [{i}/{len(factor_names)}] {factor_name}: 失败 — {e}')

    logger.info(f'[{year_month}] 完成 {ok_count}/{len(factor_names)} 个因子')


In [8]:
# ============================================================================
# 5) 主循环：逐月计算全部因子
# ============================================================================

logger.info('=' * 60)
logger.info(f'开始逐月计算因子 |  {len(years_months)} 个月 x {len(FACTORS)} 个因子')
logger.info('=' * 60)

for year, month in tqdm(years_months, desc='计算因子'):
    cal_factors_for_month(
        year=year,
        month=month,
        lab=lab,
        factor_names=FACTORS,
        index_symbol=VT_INDEX_SYMBOL,
        current_date=END,
        factor_type=DEFAULT_FACTOR_TYPE,
    )

logger.info('=' * 60)
logger.info('全部因子计算完成')
logger.info('=' * 60)


2026-05-22 18:10:41 ============================================================
2026-05-22 18:10:41 开始逐月计算因子 | 共 101 个月 x 65 个因子
2026-05-22 18:10:41 ============================================================


计算因子:   0%|          | 0/101 [00:00<?, ?it/s]

2026-05-22 18:10:41 [2018-01] 时间范围: 2018-01-01 ~ 2018-01-31
2026-05-22 18:10:41 [2018-01] 成分股数量: 300
2026-05-22 18:10:41 开始计算...
2026-05-22 18:10:44 [2018-01-01-2018-01-31] 分线数据加载完成，行数: 1584000
2026-05-22 18:10:44 计算完成
2026-05-22 18:10:44 late_skew_ret保存至D:\Aquant project\MF\MF_lab\factor\price and volume\late_skew_ret.parquet
2026-05-22 18:10:44   [1/65] late_skew_ret: 已保存
2026-05-22 18:10:44 开始计算...
2026-05-22 18:10:46 [2018-01-01-2018-01-31] 分线数据加载完成，行数: 1584000
2026-05-22 18:10:46 计算完成
2026-05-22 18:10:46 down_vol_perc保存至D:\Aquant project\MF\MF_lab\factor\price and volume\down_vol_perc.parquet
2026-05-22 18:10:46   [2/65] down_vol_perc: 已保存
2026-05-22 18:10:46 开始计算...
2026-05-22 18:10:48 [2018-01-01-2018-01-31] 分线数据加载完成，行数: 1584000
2026-05-22 18:10:49 计算完成
2026-05-22 18:10:49 corr_ret_lastret保存至D:\Aquant project\MF\MF_lab\factor\price and volume\corr_ret_lastret.parquet
2026-05-22 18:10:49   [3/65] corr_ret_lastret: 已保存
2026-05-22 18:10:49 开始计算...
2026-05-22 18:10:51 [2018-01-01-20

计算因子:   1%|          | 1/101 [01:27<2:25:46, 87.46s/it]

2026-05-22 18:12:09 [2018-02] 时间范围: 2018-02-01 ~ 2018-02-28
2026-05-22 18:12:09 [2018-02] 成分股数量: 300
2026-05-22 18:12:09 开始计算...
2026-05-22 18:12:11 [2018-02-01-2018-02-28] 分线数据加载完成，行数: 1080000
2026-05-22 18:12:11 计算完成
2026-05-22 18:12:11 late_skew_ret保存至D:\Aquant project\MF\MF_lab\factor\price and volume\late_skew_ret.parquet
2026-05-22 18:12:11   [1/65] late_skew_ret: 已保存
2026-05-22 18:12:11 开始计算...
2026-05-22 18:12:13 [2018-02-01-2018-02-28] 分线数据加载完成，行数: 1080000
2026-05-22 18:12:13 计算完成
2026-05-22 18:12:13 down_vol_perc保存至D:\Aquant project\MF\MF_lab\factor\price and volume\down_vol_perc.parquet
2026-05-22 18:12:13   [2/65] down_vol_perc: 已保存
2026-05-22 18:12:13 开始计算...
2026-05-22 18:12:15 [2018-02-01-2018-02-28] 分线数据加载完成，行数: 1080000
2026-05-22 18:12:15 计算完成
2026-05-22 18:12:15 corr_ret_lastret保存至D:\Aquant project\MF\MF_lab\factor\price and volume\corr_ret_lastret.parquet
2026-05-22 18:12:15   [3/65] corr_ret_lastret: 已保存
2026-05-22 18:12:15 开始计算...
2026-05-22 18:12:17 [2018-02-01-20

...

2026-05-22 20:36:41 [2026-04] 时间范围: 2026-04-01 ~ 2026-04-30
2026-05-22 20:36:41 [2026-04] 成分股数量: 300
2026-05-22 20:36:41 开始计算...
2026-05-22 20:36:43 [2026-04-01-2026-04-30] 分线数据加载完成，行数: 1296000
2026-05-22 20:36:43 计算完成
2026-05-22 20:36:43 late_skew_ret保存至D:\Aquant project\MF\MF_lab\factor\price and volume\late_skew_ret.parquet
2026-05-22 20:36:43   [1/65] late_skew_ret: 已保存
2026-05-22 20:36:43 开始计算...
2026-05-22 20:36:44 [2026-04-01-2026-04-30] 分线数据加载完成，行数: 1296000
2026-05-22 20:36:45 计算完成
2026-05-22 20:36:45 down_vol_perc保存至D:\Aquant project\MF\MF_lab\factor\price and volume\down_vol_perc.parquet
2026-05-22 20:36:45   [2/65] down_vol_perc: 已保存
2026-05-22 20:36:45 开始计算...
2026-05-22 20:36:47 [2026-04-01-2026-04-30] 分线数据加载完成，行数: 1296000
2026-05-22 20:36:47 计算完成
2026-05-22 20:36:47 corr_ret_lastret保存至D:\Aquant project\MF\MF_lab\factor\price and volume\corr_ret_lastret.parquet
2026-05-22 20:36:47   [3/65] corr_ret_lastret: 已保存
2026-05-22 20:36:47 开始计算...
2026-05-22 20:36:49 [2026-04-01-20

计算因子:  99%|█████████▉| 100/101 [2:27:19<01:24, 84.16s/it]

2026-05-22 20:38:01     月末截断: 2026-05-31 -> 2026-05-15
2026-05-22 20:38:01 [2026-05] 时间范围: 2026-05-01 ~ 2026-05-15
2026-05-22 20:38:01 [2026-05] 成分股数量: 300
2026-05-22 20:38:01 开始计算...
2026-05-22 20:38:01   [1/65] late_skew_ret: 失败 — cannot concat empty list
2026-05-22 20:38:01 开始计算...
2026-05-22 20:38:01   [2/65] down_vol_perc: 失败 — cannot concat empty list
2026-05-22 20:38:01 开始计算...
2026-05-22 20:38:02   [3/65] corr_ret_lastret: 失败 — cannot concat empty list
2026-05-22 20:38:02 开始计算...
2026-05-22 20:38:02   [4/65] corr_close_nextopen: 失败 — cannot concat empty list
2026-05-22 20:38:02 开始计算...
2026-05-22 20:38:02   [5/65] volume_perc2: 失败 — cannot concat empty list
2026-05-22 20:38:02 开始计算...
2026-05-22 20:38:03   [6/65] volume_perc3: 失败 — cannot concat empty list
2026-05-22 20:38:03 开始计算...
2026-05-22 20:38:03   [7/65] volume_perc4: 失败 — cannot concat empty list
2026-05-22 20:38:03 开始计算...
2026-05-22 20:38:03   [8/65] volume_perc5: 失败 — cannot concat empty list
2026-05-22 20:38:03 开始计

计算因子: 100%|██████████| 101/101 [2:27:50<00:00, 87.82s/it]

2026-05-22 20:38:32 ============================================================
2026-05-22 20:38:32 全部因子计算完成
2026-05-22 20:38:32 ============================================================


In [9]:
# ============================================================================
# 6) 【可选】验证：加载单个因子查看数据
# ============================================================================

# 示例：加载 late_skew_ret 因子 2026-03 数据
verify_req = FactorRequest(
    is_FD=True,
    start=datetime(2026, 3, 1),
    end=datetime(2026, 3, 31),
    symbols=['000001'],
    exchanges=[Exchange.SZSE],
    factor_name='late_skew_ret',
    factor_type=FactorType.PRICE_AND_VOLUME,
)

df_verify = lab.load_factor(verify_req)
logger.info(f'验证加载 shape: {df_verify.shape}')
df_verify.head()


2026-05-22 20:45:02 late_skew_ret: 成功加载1 只股票
2026-05-22 20:45:02 验证加载 shape: (22, 3)


vt_symbol,datetime,data
str,datetime[μs],f64
"""000001.SZSE""",2026-03-02 00:00:00,0.15491
"""000001.SZSE""",2026-03-03 00:00:00,-0.2579
"""000001.SZSE""",2026-03-04 00:00:00,0.077087
"""000001.SZSE""",2026-03-05 00:00:00,0.000748
"""000001.SZSE""",2026-03-06 00:00:00,0.000846


In [4]:
# ============================================================================
# 7) 【可选】缺失率检查
# ============================================================================

# 检查指定时间段内全部因子的缺失率

for factor_name in FACTORS:
    miss_req = FactorRequest(
        is_FD=True,
        start=None,
        end=None,
        symbols= None,
        exchanges= None,
        factor_name=factor_name,
        factor_type=FactorType.PRICE_AND_VOLUME,
    )
    lab.missing_ratio(miss_req)


2026-05-22 21:06:50 late_skew_ret: 成功加载553 只股票
2026-05-22 21:06:50 late_skew_ret: 总行数 623,417, null数 0 (0.0000%), NaN数 0 (0.0000%)
2026-05-22 21:06:50 down_vol_perc: 成功加载553 只股票
2026-05-22 21:06:50 down_vol_perc: 总行数 611,090, null数 3,818 (0.6248%), NaN数 0 (0.0000%)
2026-05-22 21:06:50 corr_ret_lastret: 成功加载553 只股票
2026-05-22 21:06:50 corr_ret_lastret: 总行数 611,090, null数 0 (0.0000%), NaN数 3,807 (0.6230%)
2026-05-22 21:06:50 corr_close_nextopen: 成功加载553 只股票
2026-05-22 21:06:50 corr_close_nextopen: 总行数 611,090, null数 0 (0.0000%), NaN数 565 (0.0925%)
2026-05-22 21:06:51 volume_perc2: 成功加载553 只股票
2026-05-22 21:06:51 volume_perc2: 总行数 611,090, null数 0 (0.0000%), NaN数 0 (0.0000%)
2026-05-22 21:06:51 volume_perc3: 成功加载553 只股票
2026-05-22 21:06:51 volume_perc3: 总行数 611,090, null数 0 (0.0000%), NaN数 0 (0.0000%)
2026-05-22 21:06:51 volume_perc4: 成功加载553 只股票
2026-05-22 21:06:51 volume_perc4: 总行数 611,090, null数 0 (0.0000%), NaN数 0 (0.0000%)
2026-05-22 21:06:51 volume_perc5: 成功加载553 只股票
2026-05-22 21:0